# 01 · Workflow justification (REQ-02)

**Question:** is transaction-dispute intake the workflow with the most evidence and value? (§8.1)
Exit criterion: if disputes/transactions are marginal compared with another contact reason → Card-Service Support.

Output: `analysis/results/workflow_justification.json` (aggregates) + figures in `analysis/figures/`.

1. Contact reasons and volume · 2. Demand patterns · 3. Disputes within complaints ·
4. Linking complaints to transactions (§14 risk) · 5. Data quality · 6. Operational constraints · 7. Conclusion

In [ ]:
from common import ROOT, connect, save_result, show

con = connect()

## 1. Contact reasons and volume

In [ ]:
reasons = show(con, "Interactions by category (+ escalation, resolution)", """
    SELECT reason_category, count(*) AS n, round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct,
           round(avg(was_escalated::INT), 3) AS p_escalado, round(avg(was_resolved::INT), 3) AS p_resuelto,
           round(median(duration_seconds)) AS dur_p50_s
    FROM call_center_interactions GROUP BY 1 ORDER BY n DESC""")

## 2. Demand patterns (monthly, by country/channel)

In [ ]:
monthly = show(con, "Complaints by month and category", """
    SELECT date_trunc('month', creation_date::TIMESTAMP) AS mes, category, count(*) AS n
    FROM complaints GROUP BY ALL ORDER BY mes, category""", 60)

## 3. Disputes within complaints
`Cargo no reconocido` (unrecognized charge) + `Cobro indebido` (improper charge) = the transaction disputes in our scope.

In [ ]:
disputes = show(con, "Complaint subcategories", """
    SELECT subcategory, count(*) AS n, round(100.0 * count(*) / sum(count(*)) OVER (), 1) AS pct,
           round(avg(sla_breached::INT), 3) AS p_sla_roto, round(avg(is_repeat_complainer::INT), 3) AS p_reincidente,
           round(median(resolution_days), 1) AS dias_p50
    FROM complaints GROUP BY 1 ORDER BY n DESC""")

## 4. Can complaints be linked to transactions? (§14)

No. The affected product never belongs to the complainant and the claimed amount matches their transactions at chance
level (details and figures in `02_policy_calibration`). A dispute is anchored to the transaction the customer
identifies in the conversation and the API verifies as theirs (POL-1), not to a historical link in the dataset.

In [ ]:
import json

calibration = json.loads((ROOT / "analysis" / "results" / "policy_calibration.json").read_text(encoding="utf-8"))
link = calibration["dispute_transaction_link"]
link

## 5. Data quality
Nulls, duplicates and orphans in the workflow's columns (complements OPS's `data/_state/reports/latest.json`).

In [ ]:
show(con, "Nulls in key complaint columns", """
    SELECT round(avg((claimed_amount IS NULL)::INT), 3) AS claimed_amount_nulo,
           round(avg((affected_product_id IS NULL)::INT), 3) AS producto_nulo,
           round(avg((subcategory IS NULL)::INT), 3) AS subcategoria_nula,
           count(*) - count(DISTINCT complaint_id) AS duplicados
    FROM complaints""")

## 6. Operational constraints
SLA, resolution times and load per agent: which part can be automated without moving money (CON-05).

In [ ]:
ops = show(con, "Disputes vs other complaints: SLA, resolution days, first response", """
    SELECT CASE WHEN subcategory IN ('Cargo no reconocido', 'Cobro indebido') THEN 'dispute'
                WHEN subcategory IS NULL THEN 'no subcategory' ELSE 'other' END AS grupo,
           count(*) AS n, round(avg(sla_breached::INT), 3) AS p_sla_roto,
           round(median(resolution_days), 1) AS dias_p50, round(quantile_cont(resolution_days, 0.9), 1) AS dias_p90,
           round(median(datediff('hour', TRY_CAST(creation_date AS TIMESTAMP),
                                 TRY_CAST(first_response_date AS TIMESTAMP))), 1) AS horas_primera_respuesta_p50,
           count(DISTINCT assigned_agent_id) AS agentes
    FROM complaints GROUP BY 1 ORDER BY n DESC""")
by_priority = show(con, "Disputes by priority", """
    SELECT priority, count(*) AS n, round(avg(sla_breached::INT), 3) AS p_sla_roto,
           round(median(resolution_days), 1) AS dias_p50
    FROM complaints WHERE subcategory IN ('Cargo no reconocido', 'Cobro indebido')
    GROUP BY 1 ORDER BY n DESC""")


## 7. Conclusion
Transaction disputes are the largest homogeneous slice of complaints (about a third of the total, two subcategories
with the same intake), they breach SLA as often as the rest, and they wait days for a first human response and weeks
for resolution. The intake step (identify the transaction, check eligibility, register the dispute) moves no money
(CON-05), so it can be automated safely; the investigation and any refund stay with people. The exit criterion
(disputes marginal against another contact reason) does not hold, so we keep the dispute-intake workflow.


In [ ]:
n_total = int(disputes["n"].sum())
dispute_rows = disputes.filter(disputes["subcategory"].is_in(["Cargo no reconocido", "Cobro indebido"]))
dispute_share_pct = round(100 * int(dispute_rows["n"].sum()) / n_total, 1)
dispute_ops = ops.filter(ops["grupo"] == "dispute").to_dicts()[0]
exit_criterion_met = dispute_share_pct < 10  # "marginal": below a tenth of complaints
decision = (
    f"Keep dispute intake: disputes are {dispute_share_pct}% of complaints ({int(dispute_rows['n'].sum()):,} of "
    f"{n_total:,}); SLA breached in {dispute_ops['p_sla_roto']:.1%}, first response p50 "
    f"{dispute_ops['horas_primera_respuesta_p50']:.0f} h, resolution p50 {dispute_ops['dias_p50']:.0f} days. "
    "Intake moves no money (CON-05). Exit criterion (disputes marginal) not met."
)
assert not exit_criterion_met
print(decision)
save_result("workflow_justification", {
    "reason_categories": reasons.to_dicts(),
    "complaint_subcategories": disputes.to_dicts(),
    "dispute_share_pct": dispute_share_pct,
    "dispute_n": int(dispute_rows["n"].sum()),
    "complaints_n": n_total,
    "operational_constraints": ops.to_dicts(),
    "disputes_by_priority": by_priority.to_dicts(),
    "exit_criterion_met": exit_criterion_met,
    "decision": decision,
})
